In [1]:
"""
================================================================================
STEP 3: TRAIN GCN ON GRAPH
================================================================================
Trains a Graph Convolutional Network using:
- BERT embeddings as node features
- kNN graph structure
- Semi-supervised learning (10% labeled)
================================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
import json

print("="*80)
print("TRAINING GCN")
print("="*80)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TRAINING GCN
Device: cuda


In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    # Data files
    EMBEDDINGS_PATH = 'node_features.npy'
    LABELS_PATH = 'labels.npy'
    ADJ_PATH = 'adjacency_matrix.npz'
    TRAIN_MASK_PATH = 'train_mask.npy'
    VAL_MASK_PATH = 'val_mask.npy'
    TEST_MASK_PATH = 'test_mask.npy'
    
    # GCN settings
    HIDDEN_DIM = 256
    DROPOUT = 0.5
    LEARNING_RATE = 0.01
    WEIGHT_DECAY = 5e-4
    EPOCHS = 200
    EARLY_STOPPING_PATIENCE = 20
    
    # Output
    MODEL_SAVE_PATH = 'best_gcn_model.pt'
    RESULTS_PATH = 'gcn_results.json'

config = Config()

In [3]:
# ============================================================================
# LOAD DATA
# ============================================================================
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

# Load embeddings
embeddings = np.load(config.EMBEDDINGS_PATH)
print(f"✓ Embeddings: {embeddings.shape}")

# Load labels
labels = np.load(config.LABELS_PATH)
num_classes = len(np.unique(labels))
print(f"✓ Labels: {labels.shape}, Classes: {num_classes}")

# Load masks
train_mask = np.load(config.TRAIN_MASK_PATH)
val_mask = np.load(config.VAL_MASK_PATH)
test_mask = np.load(config.TEST_MASK_PATH)
print(f"✓ Train: {train_mask.sum():,}, Val: {val_mask.sum():,}, Test: {test_mask.sum():,}")

# Load adjacency matrix
adj = sp.load_npz(config.ADJ_PATH)
print(f"✓ Adjacency: {adj.shape}, Edges: {adj.nnz:,}")


LOADING DATA
✓ Embeddings: (122817, 768)
✓ Labels: (122817,), Classes: 10
✓ Train: 12,281, Val: 6,141, Test: 104,395
✓ Adjacency: (122817, 122817), Edges: 2,209,333


In [4]:
# ============================================================================
# PREPROCESSING
# ============================================================================
print("\n" + "="*80)
print("PREPROCESSING GRAPH")
print("="*80)

def normalize_adj(adj):
    """
    Symmetrically normalize adjacency matrix: D^(-1/2) * A * D^(-1/2)
    Note: Self-loops are already included from graph building
    """
    # Self-loops already added in step2_build_graph.py, no need to add again
    rowsum = np.array(adj.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    return d_mat_inv_sqrt @ adj @ d_mat_inv_sqrt

def sparse_to_torch(sparse_mx):
    """Convert scipy sparse matrix to torch sparse tensor"""
    sparse_mx = sparse_mx.tocoo()
    indices = torch.from_numpy(
        np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64)
    )
    values = torch.from_numpy(sparse_mx.data.astype(np.float32))
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse_coo_tensor(indices, values, shape)

# Normalize adjacency
adj_normalized = normalize_adj(adj)
print("✓ Normalized adjacency matrix")

# Convert to PyTorch
features = torch.FloatTensor(embeddings).to(device)
labels_tensor = torch.LongTensor(labels).to(device)
train_mask_tensor = torch.BoolTensor(train_mask).to(device)
val_mask_tensor = torch.BoolTensor(val_mask).to(device)
test_mask_tensor = torch.BoolTensor(test_mask).to(device)
adj_normalized = sparse_to_torch(adj_normalized).to(device)

print(f"✓ Converted to PyTorch tensors")
print(f"  Features: {features.shape}")
print(f"  Adjacency: {adj_normalized.shape}")


PREPROCESSING GRAPH
✓ Normalized adjacency matrix
✓ Converted to PyTorch tensors
  Features: torch.Size([122817, 768])
  Adjacency: torch.Size([122817, 122817])


In [5]:
# ============================================================================
# GCN MODEL
# ============================================================================
print("\n" + "="*80)
print("DEFINING GCN MODEL")
print("="*80)

class GraphConvolution(nn.Module):
    """Single graph convolution layer"""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))
        self.reset_parameters()
    
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)
    
    def forward(self, x, adj):
        support = torch.mm(x, self.weight)
        output = torch.sparse.mm(adj, support)
        return output + self.bias

class GCN(nn.Module):
    """Two-layer Graph Convolutional Network"""
    def __init__(self, n_features, n_hidden, n_classes, dropout=0.5):
        super().__init__()
        self.gc1 = GraphConvolution(n_features, n_hidden)
        self.gc2 = GraphConvolution(n_hidden, n_classes)
        self.dropout = dropout
    
    def forward(self, x, adj):
        x = F.relu(self.gc1(x, adj))
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.gc2(x, adj)
        return F.log_softmax(x, dim=1)

# Initialize model
n_features = embeddings.shape[1]
model = GCN(
    n_features=n_features,
    n_hidden=config.HIDDEN_DIM,
    n_classes=num_classes,
    dropout=config.DROPOUT
).to(device)

print(f"✓ GCN Model:")
print(f"  Input: {n_features} (BERT embeddings)")
print(f"  Hidden: {config.HIDDEN_DIM}")
print(f"  Output: {num_classes} (categories)")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)



DEFINING GCN MODEL
✓ GCN Model:
  Input: 768 (BERT embeddings)
  Hidden: 256
  Output: 10 (categories)
  Parameters: 199,434


In [6]:
# ============================================================================
# TRAINING
# ============================================================================
print("\n" + "="*80)
print("TRAINING GCN")
print("="*80)

best_val_f1 = 0.0
best_epoch = 0
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}

for epoch in range(config.EPOCHS):
    # Train
    model.train()
    optimizer.zero_grad()
    output = model(features, adj_normalized)
    
    # Loss only on training nodes
    loss = F.nll_loss(output[train_mask_tensor], labels_tensor[train_mask_tensor])
    loss.backward()
    optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        output = model(features, adj_normalized)
        
        # Train metrics
        train_pred = output[train_mask_tensor].argmax(dim=1)
        train_true = labels_tensor[train_mask_tensor]
        train_acc = accuracy_score(train_true.cpu(), train_pred.cpu())
        
        # Val metrics
        val_pred = output[val_mask_tensor].argmax(dim=1)
        val_true = labels_tensor[val_mask_tensor]
        val_acc = accuracy_score(val_true.cpu(), val_pred.cpu())
        val_f1 = f1_score(val_true.cpu(), val_pred.cpu(), average='weighted')
    
    history['train_loss'].append(loss.item())
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{config.EPOCHS} | '
              f'Loss: {loss.item():.4f} | '
              f'Train Acc: {train_acc:.4f} | '
              f'Val Acc: {val_acc:.4f} | '
              f'Val F1: {val_f1:.4f}')
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc
        }, config.MODEL_SAVE_PATH)
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= config.EARLY_STOPPING_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        print(f'Best val F1: {best_val_f1:.4f} at epoch {best_epoch+1}')
        break



TRAINING GCN
Epoch  10/200 | Loss: 0.2117 | Train Acc: 0.9564 | Val Acc: 0.8832 | Val F1: 0.8832
Epoch  20/200 | Loss: 0.2180 | Train Acc: 0.9571 | Val Acc: 0.8771 | Val F1: 0.8769
Epoch  30/200 | Loss: 0.1915 | Train Acc: 0.9570 | Val Acc: 0.8798 | Val F1: 0.8798

Early stopping at epoch 31
Best val F1: 0.8837 at epoch 11


In [7]:
# ============================================================================
# TESTING
# ============================================================================
print("\n" + "="*80)
print("TESTING")
print("="*80)

# Load best model
checkpoint = torch.load(config.MODEL_SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best model from epoch {checkpoint['epoch']+1}")

# Test
model.eval()
with torch.no_grad():
    output = model(features, adj_normalized)
    test_pred = output[test_mask_tensor].argmax(dim=1)
    test_true = labels_tensor[test_mask_tensor]
    
    test_acc = accuracy_score(test_true.cpu(), test_pred.cpu())
    test_f1 = f1_score(test_true.cpu(), test_pred.cpu(), average='weighted')

print(f"\n{'='*70}")
print(f"GCN FINAL TEST RESULTS")
print(f"{'='*70}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"{'='*70}")

# Detailed report
from sklearn.preprocessing import LabelEncoder
# Load class names from metadata if available
try:
    with open('embedding_metadata.json', 'r') as f:
        metadata = json.load(f)
        class_names = metadata['class_names']
except:
    class_names = [str(i) for i in range(num_classes)]

print("\nClassification Report:")
print(classification_report(
    test_true.cpu(),
    test_pred.cpu(),
    target_names=class_names,
    digits=4
))

# Save results
results = {
    'best_epoch': int(checkpoint['epoch'] + 1),
    'test_accuracy': float(test_acc),
    'test_f1': float(test_f1),
    'val_f1': float(checkpoint['val_f1']),
    'history': history,
    'config': {
        'hidden_dim': config.HIDDEN_DIM,
        'dropout': config.DROPOUT,
        'learning_rate': config.LEARNING_RATE,
        'weight_decay': config.WEIGHT_DECAY
    }
}

with open(config.RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to: {config.RESULTS_PATH}")
print("="*80)



TESTING
✓ Loaded best model from epoch 11

GCN FINAL TEST RESULTS
Test Accuracy: 0.8793
Test F1 Score: 0.8793

Classification Report:
                        precision    recall  f1-score   support

                   art     0.7772    0.7676    0.7724      8349
                 crime     0.9415    0.9368    0.9391     12742
               economy     0.8743    0.8747    0.8745     12244
             education     0.9032    0.8993    0.9012      7540
         entertainment     0.8145    0.8177    0.8161     10024
                global     0.8767    0.8780    0.8774     12296
                health     0.8620    0.8777    0.8698     12353
              politics     0.9110    0.8789    0.8947     12561
science and technology     0.8302    0.8436    0.8369      6734
                sports     0.9602    0.9845    0.9722      9552

              accuracy                         0.8793    104395
             macro avg     0.8751    0.8759    0.8754    104395
          weighted avg     0.87